In [13]:
%useLatestDescriptors
%use serialization, kandy

In [14]:
@Serializable
public data class JmhReport(
    val jmhVersion: String,
    val benchmark: String,
    val mode: String,
    val threads: UInt,
    val forks: UInt,
    val jvm: String,
    val jvmArgs: List<String>,
    val jdkVersion: String,
    val vmName: String,
    val vmVersion: String,
    val warmupIterations: UInt,
    val warmupTime: String,
    val warmupBatchSize: UInt,
    val measurementIterations: UInt,
    val measurementTime: String,
    val measurementBatchSize: UInt,
    val params: Map<String, String> = emptyMap(),
    val primaryMetric: PrimaryMetric,
    val secondaryMetrics: Map<String, SecondaryMetric>,
) {
    public interface Metric {
        public val score: Double
        public val scoreError: Double
        public val scoreConfidence: List<Double>
        public val scorePercentiles: Map<Double, Double>
        public val scoreUnit: String
    }

    @Serializable
    public data class PrimaryMetric(
        override val score: Double,
        override val scoreError: Double,
        override val scoreConfidence: List<Double>,
        override val scorePercentiles: Map<Double, Double>,
        override val scoreUnit: String,
        val rawDataHistogram: List<List<List<List<Double>>>>? = null,
        val rawData: List<List<Double>>? = null,
    ) : Metric

    @Serializable
    public data class SecondaryMetric(
        override val score: Double,
        override val scoreError: Double,
        override val scoreConfidence: List<Double>,
        override val scorePercentiles: Map<Double, Double>,
        override val scoreUnit: String,
        val rawData: List<List<Double>>,
    ) : Metric
}

In [15]:
import java.io.File

@OptIn(ExperimentalSerializationApi::class)
val reports = Json.decodeFromStream<List<JmhReport>>(File("data/arrayAllocation-1.json").inputStream())

In [114]:
val reportsByName = reports.groupBy { it.benchmark }.mapValues { it.value.sortedBy { it.params["size"]!!.toInt() } }
val dataByName = reportsByName.mapValues {
    val value = it.value
    mapOf(
//        "size" to value.map { it.params["size"]!!.toInt().toDouble() },
        "size" to value.indices.map { it.toDouble() },
        "score" to value.map { it.primaryMetric.score },
        "error" to value.map { it.primaryMetric.scoreError },
    )
}

val plots = dataByName.mapValues { (name, data) ->
    val xs = data["size"]!!
    val scores = data["score"]!!
    val errors = data["error"]!!
    val yMins = scores.zip(errors) { score, error -> log2(score - error) }
    val yMaxs = scores.zip(errors) { score, error -> log2(score + error) }
    val minShift = yMins.zip(xs) { y, x -> y - x }.min()
    val maxShift = yMaxs.zip(xs) { y, x -> y - x }.max()
    plot(data) {
        layout.title = name.substringAfter("dev.lounres.kone.benchmarks.collections.")
        x(xs, name = "log2(size)")
        y.axis.name = "log2(time)"
        errorBars {
            yMin(yMins)
            yMax(yMaxs)
        }
        line {
            y(xs.map { it + minShift })
        }
        line {
            y(xs.map { it + maxShift })
        }
    }
}

plotGrid(plots.values.toList(), nCol = 2)
//reportsByName.mapValues { it.value.last().primaryMetric.score }.values.max() / 10.0.pow(9)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="4btjCv"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"layout":{
"name":"grid",
"ncol":2,
"nrow":5,
"fit":true,
"align":false
},
"figures":[{
"ggtitle":{
"text":"array.ArrayAllocationBenchmarks.kone_boolean"
},
"mapping":{
},
"data":{
"ymin":[2.1511565450488863,2.0266022919213604,2.2902088737918267,3.1414953613937437,3.556238716928153,4.005521333403118,4.60168452624909,5.353882366360512,6.242112166982028,7.426406825105354,8.354829025321646,9.390832416205477,10.37540739371857,11.375559255156244,12.363577746512407,13.373998380592964,14.416627885566982,15.370936414558367,16.353192168716657,17.329320576201866,18.322034603609573,19.3591882001252,20.159157485564894,21.158189241369374,22.15390934712975,23.157997623366903,24.14588894921529,25.1442565301268,26.15281007960304,27.148903288165975,28.135230927397565],
"y*":[2.195543605973932,3.195543605973932,4.195543605973932,5.195543605973932,6.195543605973932,7.195543605973932,8.195543605973931,9.195543605973931,10.195543605973931,11.195543605973931,12.195543605973931,13.195543605973931,14.195543605973931,15.195543605973931,16.19554360597393,17.19554360597393,18.19554360597393,19.19554360597393,20.19554360597393,21.19554360597393,22.19554360597393,23.19554360597393,24.19554360597393,25.19554360597393,26.19554360597393,27.19554360597393,28.19554360597393,29.19554360597393,30.19554360597393,31.19554360597393,32.19554360597393],
"ymax":[2.195543605973932,2.0719001491945326,2.3236757953757423,3.1610014784714076,3.574955147384767,4.022515616590426,4.610779807164036,5.359364372640832,6.245559829622485,7.431097860315223,8.36894790712673,9.39835202796919,10.377031358621036,11.377167631992434,12.371877286655964,13.388262648671535,14.427872553657707,15.385675635813277,16.361423932881685,17.340793189193676,18.346680424580615,19.363321787850644,20.16541617638857,21.171231246127334,22.158280483420207,23.164251882606603,24.156201778650757,25.157145345816872,26.167221891708774,27.15472251706551,28.145946940566088],
"y":[-1.864769072602435,-0.8647690726024351,0.13523092739756493,1.135230927397565,2.135230927397565,3.135230927397565,4.135230927397565,5.135230927397565,6.135230927397565,7.135230927397565,8.135230927397565,9.135230927397565,10.135230927397565,11.135230927397565,12.135230927397565,13.135230927397565,14.135230927397565,15.135230927397565,16.135230927397565,17.135230927397565,18.135230927397565,19.135230927397565,20.135230927397565,21.135230927397565,22.135230927397565,23.135230927397565,24.135230927397565,25.135230927397565,26.135230927397565,27.135230927397565,28.135230927397565],
"log2(size)":[0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"log2(time)",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"log2(size)",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
},{
"mapping":{
"x":"log2(size)",
"y":"y"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data":{
}
},{
"mapping":{
"x":"log2(size)",
"y":"y*"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"log2(size)"
},{
"type":"float",
"column":"ymin"
},{
"

In [115]:
val start = 2

val data = buildList {
    addAll(1 ..< (1 shl start))
    for (i in start .. 29) {
        addAll((1 shl i) ..< (1 shl (i + 1)) step (1 shl (i - start)))
    }
}

println(data.size)

plot {
    x(data.map { log2(it.toDouble()) })
    points {
        y(data.map { log2(it.toDouble()) })
    }
}

115


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Swio3Q"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"x":[0.0,1.0,1.5849625007211563,2.0,2.321928094887362,2.584962500721156,2.807354922057604,3.0,3.3219280948873626,3.5849625007211565,3.8073549220576037,4.0,4.321928094887363,4.584962500721157,4.807354922057604,5.0,5.321928094887363,5.584962500721157,5.807354922057605,6.0,6.321928094887362,6.584962500721156,6.807354922057604,7.0,7.321928094887362,7.584962500721156,7.807354922057604,8.0,8.321928094887362,8.584962500721156,8.807354922057604,9.0,9.321928094887362,9.584962500721156,9.807354922057604,10.0,10.321928094887362,10.584962500721156,10.807354922057604,11.0,11.321928094887364,11.584962500721158,11.807354922057604,12.0,12.321928094887364,12.584962500721158,12.807354922057604,13.0,13.321928094887364,13.584962500721158,13.807354922057604,14.0,14.321928094887364,14.584962500721158,14.807354922057604,15.0,15.321928094887364,15.584962500721158,15.807354922057604,16.0,16.321928094887365,16.584962500721158,16.807354922057606,17.0,17.321928094887365,17.584962500721158,17.807354922057606,18.0,18.321928094887365,18.584962500721154,18.807354922057606,19.0,19.321928094887365,19.584962500721154,19.807354922057606,20.0,20.321928094887365,20.584962500721154,20.807354922057606,21.0,21.32192809488736,21.584962500721154,21.807354922057606,22.0,22.32192809488736,22.584962500721158,22.807354922057606,23.0,23.32192809488736,23.584962500721158,23.807354922057602,24.0,24.321928094887365,24.584962500721158,24.807354922057606,25.0,25.32192809488736,25.584962500721158,25.807354922057602,26.0,26.321928094887365,26.584962500721158,26.807354922057606,27.0,27.32192809488736,27.584962500721158,27.807354922057602,28.0,28.321928094887365,28.584962500721154,28.807354922057606,29.000000000000004,29.32192809488736,29.584962500721158,29.807354922057602],
"y":[0.0,1.0,1.5849625007211563,2.0,2.321928094887362,2.584962500721156,2.807354922057604,3.0,3.3219280948873626,3.5849625007211565,3.8073549220576037,4.0,4.321928094887363,4.584962500721157,4.807354922057604,5.0,5.321928094887363,5.584962500721157,5.807354922057605,6.0,6.321928094887362,6.584962500721156,6.807354922057604,7.0,7.321928094887362,7.584962500721156,7.807354922057604,8.0,8.321928094887362,8.584962500721156,8.807354922057604,9.0,9.321928094887362,9.584962500721156,9.807354922057604,10.0,10.321928094887362,10.584962500721156,10.807354922057604,11.0,11.321928094887364,11.584962500721158,11.807354922057604,12.0,12.321928094887364,12.584962500721158,12.807354922057604,13.0,13.321928094887364,13.584962500721158,13.807354922057604,14.0,14.321928094887364,14.584962500721158,14.807354922057604,15.0,15.321928094887364,15.584962500721158,15.807354922057604,16.0,16.321928094887365,16.584962500721158,16.807354922057606,17.0,17.321928094887365,17.584962500721158,17.807354922057606,18.0,18.321928094887365,18.584962500721154,18.807354922057606,19.0,19.321928094887365,19.584962500721154,19.807354922057606,20.0,20.321928094887365,20.584962500721154,20.807354922057606,21.0,21.32192809488736,21.584962500721154,21.807354922057606,22.0,22.32192809488736,22.584962500721158,22.807354922057606,23.0,23.32192809488736,23.584962500721158,23.807354922057602,24.0,24.321928094887365,24.584962500721158,24.807354922057606,25.0,25.32192809488736,25.584962500721158,25.807354922057602,26.0,26.321928094887365,26.584962500721158,26.807354922057606,27.0,27.32192809488736,27.584962500721158,27.807354922057602,28.0,28.321928094887365,28.584962500721154,28.807354922057606,29.000000000000004,29.32192809488736,29.584962500721158,29.807354922057602]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,n